# Water–isobutanol solved-state actions

**Evidence: numerical verification.** Differentiate log activity of isobutanol in the water-rich phase.


The canonical input is the neutral water/isobutanol subset of
`khudaida-2026-figure-2-electrolyte-lle/7`, at 293.15 K and 100,000 Pa.
Khudaida et al. (2026), Tables 2, 5–7
([paper](https://doi.org/10.1021/acs.jced.5c00780)), and the packet's cited
association rules own the inputs. Version 7 retains the Wolbach–Sandler
unlike association size factor. The feed is 0.6 mol water + 0.4 mol isobutanol.


## Read the inputs

Use a repository checkout with the environment described in the installation
guide. The wheel alone does not contain these data. All density, association,
equilibrium and parameter-fit iterations below run in the native Engine.


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from epcsaft import (
    Parameters,
    Mixture,
)
from epcsaft import equilibrium as eq, regression as rg

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

ROOT = next(
    folder
    for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "ARCHITECTURE.yaml").is_file()
)
PACKETS = ROOT / "data" / "packets"


def equilibrium_values(central_state):
    """Keep the returned physical state and the equation residuals together."""
    assert central_state.success, central_state.message
    assert central_state.requested_tolerance_met
    assert central_state.classified_residuals
    residual = max(map(abs, central_state.classified_residuals), default=0)
    assert residual <= 1e-10
    return dict(
        temperature_K=central_state.temperature,
        pressure_Pa=central_state.pressure,
        amounts_mol=central_state.amounts,
        mole_fractions=central_state.mole_fractions,
        molar_densities_mol_m3=central_state.molar_densities,
        residual=residual,
        linear_solver=central_state.linear_solver,
    )


## Compile once and request exact actions

Activate the isobutanol a/b self-association energy divided by Boltzmann's
constant, measured in K. The two directions are **1 K temperature** and
**1 K association energy/k** increments. Solve the central state, compile the
same defining equations, then request first contractions and one mixed
contraction. Each contraction of log activity is dimensionless because its
path variable is dimensionless.

Divide by the physical increments to obtain partial derivatives in K⁻¹ and
K⁻². Re-solve at T ± 0.01 K and T ± 0.005 K only to verify the exact
temperature partial. Require the refined difference below 10⁻⁷ K⁻¹.
The native central and perturbed solves must retain residuals at most 10⁻¹⁰.


In [2]:
parameters = Parameters.from_json(
    PACKETS / "khudaida-2026-figure-2-electrolyte-lle/7/parameters/parameters.json",
    components=["water", "isobutanol"],
)
pair = rg.active_parameter(
    parameters, "association_energy_over_k", (("isobutanol", "a"), ("isobutanol", "b"))
)
model = Mixture(parameters, active_parameters=[pair])
problem = eq.Problem(
    T=293.15,
    P=100000.0,
    feed=eq.Amounts({"water": 0.6, "isobutanol": 0.4}),
    phases=[
        eq.Phase("Lw", kind="liquid", composition_guess=(0.999, 0.001)),
        eq.Phase("Lo", kind="liquid", composition_guess=(0.3, 0.7)),
    ],
)
central_state = eq.solve_equilibrium(model, problem)
compiled_problem = eq.compile_problem(model, problem)
observables = [
    eq.SolvedStateObservable(eq.SolvedStateObservableKind.PhaseLogActivity, 0, 1)
]
directions = [
    eq.SolvedStateActionDirection(1, 0, [0, 0], [0]),
    eq.SolvedStateActionDirection(0, 0, [0, 0], [1]),
]
gradient = eq.solved_state_first_actions(
    compiled_problem, central_state, observables, directions
)
mixed = eq.solved_state_actions(
    compiled_problem,
    central_state,
    eq.SolvedStateActionRequest(observables, directions),
)
assert all(
    (
        b.results[0].status == eq.SolvedStateActionStatus.Available
        for b in [*gradient, mixed]
    )
)


def value(t):
    declaration = eq.Problem(T=t, P=100000.0, feed=problem.feed, phases=problem.phases)
    perturbed_state = eq.solve_equilibrium(model, declaration)
    equilibrium_values(perturbed_state)
    perturbed_actions = eq.solved_state_first_actions(
        eq.compile_problem(model, declaration),
        perturbed_state,
        observables,
        [directions[0]],
    )[0]
    assert perturbed_actions.results[0].status == eq.SolvedStateActionStatus.Available
    return perturbed_actions.results[0].value


differences = [(value(293.15 + h) - value(293.15 - h)) / (2 * h) for h in (0.01, 0.005)]
temperature_increment_K = 1.0
parameter_increment_K = 1.0
temperature_partial = gradient[0].results[0].action / temperature_increment_K
parameter_partial = gradient[1].results[0].action / parameter_increment_K
mixed_partial = mixed.results[0].action / (
    temperature_increment_K * parameter_increment_K
)
assert abs(differences[1] - temperature_partial) < 1e-07
values = dict(
    equilibrium=equilibrium_values(central_state),
    log_activity=gradient[0].results[0].value,
    first_actions=[b.results[0].action for b in gradient],
    mixed_action=mixed.results[0].action,
    temperature_difference_checks=differences,
)
values["temperature_partial_K^-1"] = temperature_partial
values["parameter_partial_K^-1"] = parameter_partial
values["mixed_partial_K^-2"] = mixed_partial
values["finite_difference_steps_K"] = [0.01, 0.005]
values["refined_error_K^-1"] = differences[1] - temperature_partial
values["rank"] = mixed.diagnostics.rank
values["dimension"] = mixed.diagnostics.state_dimension
values["linear_residual"] = mixed.diagnostics.maximum_linear_residual
display(
    pd.DataFrame(
        {
            "Quantity": [
                "ln activity",
                "Temperature action",
                "Association-energy action",
                "Mixed action",
                "Temperature partial / K^-1",
                "Association-energy partial / K^-1",
                "Mixed partial / K^-2",
            ],
            "Value": [
                values["log_activity"],
                *values["first_actions"],
                values["mixed_action"],
                temperature_partial,
                parameter_partial,
                mixed_partial,
            ],
        }
    )
)


,Quantity,Value
0,ln activity,-1.329822
1,Temperature action,0.067119
2,Association-energy action,-0.001595
3,Mixed action,0.000017
4,Temperature partial / K^-1,0.067119
5,Association-energy partial / K^-1,-0.001595
6,Mixed partial / K^-2,0.000017


In [3]:
display(pd.DataFrame([values["equilibrium"]]).rename(columns={
            "temperature_K": "Temperature / K",
            "pressure_Pa": "Pressure / Pa",
            "amounts_mol": "Phase amounts / mol",
            "mole_fractions": "Mole fractions / 1",
            "molar_densities_mol_m3": "Phase densities / mol m^-3",
            "residual": "Maximum classified residual / 1",
            "linear_solver": "Linear solver",
        }))
display(pd.DataFrame({
    "Finite-difference step / K": values["finite_difference_steps_K"],
    "Temperature partial check / K^-1": values["temperature_difference_checks"],
}))
display(pd.DataFrame([{
    "Refined difference error / K^-1": values["refined_error_K^-1"],
    "Rank / 1": values["rank"],
    "State dimension / 1": values["dimension"],
    "Maximum linear residual / 1": values["linear_residual"],
}]))


,Temperature / K,Pressure / Pa,Phase amounts / mol,Mole fractions / 1,Phase densities / mol m^-3,Maximum classified residual / 1,Linear solver
0,293.15,100000.0,"[0.2853182931107769, 0.7146817068892232]","[0.9943748683039002, 0.005625131696099895, 0.4425559753585558, 0.5574440246414442]","[54198.3600462764, 16912.29929663858]",2.073248e-14,


,Finite-difference step / K,Temperature partial check / K^-1
0,0.010,0.067119
1,0.005,0.067119


,Refined difference error / K^-1,Rank / 1,State dimension / 1,Maximum linear residual / 1
0,3.765263e-11,10,10,1.242394e-16


## Meaning and limits


These derivatives apply to the smooth declared branch with unchanged supports
and distinct liquid phases. Singular, coalesced or disappearing phases need
their returned diagnostic rather than a substituted derivative. Finite
differences verify exact native actions here; they do not supply derivatives
for an optimizer or extend the source validity range. No physical validation
or parameter-uncertainty estimate follows from the derivative comparison.
